# Notebook 03 — 配对负样本 + 合并建模数据集

**输入**
- `data/processed/01_positive_samples.csv`：138 家正样本
- `data/processed/02_financials_long.csv`：全 A 股财务数据（5105 家 × 3 年）

**输出**
- `data/processed/03_modeling_dataset.csv`：正负样本合并后的建模数据集（约 276 行 × 40 列）

## 配对逻辑
每家正样本公司，在**同一年份、同申万行业、总资产规模 ±30%** 的候选池里随机抽 1 家负样本。  
负样本候选池排除正样本 138 家（方案 B：快速版）。

| Step | 内容 |
|------|------|
| 1 | 加载数据，检查正样本覆盖率 |
| 2 | 构建配对函数 |
| 3 | 执行 1:1 配对 |
| 4 | 合并正负样本，保存 CSV |
| 5 | 质量检查报告 |

In [10]:
import pandas as pd
import numpy as np
from pathlib import Path

PROCESSED_DIR = Path("..").resolve() / "data" / "processed"

# 加载正样本
pos = pd.read_csv(PROCESSED_DIR / "01_positive_samples.csv", encoding="utf-8-sig")

# 加载全 A 股财务数据
fin = pd.read_csv(PROCESSED_DIR / "02_financials_long.csv", encoding="utf-8-sig")

print(f"正样本: {len(pos)} 家")
print(f"财务数据: {fin['stock_code'].nunique()} 家公司, {len(fin)} 行")

正样本: 138 家
财务数据: 5105 家公司, 15318 行


## Step 1 — 检查正样本覆盖率

正样本的财务数据必须在 `02_financials_long.csv` 里能找到，才能参与建模。  
用 `stock_code + t_minus_2_year` 做联合键，把正样本和财务数据拼起来，看有多少能匹配上。

In [11]:
# 把正样本和财务数据拼起来
# 匹配条件：股票代码相同 + 财务年份 = T-2 年
pos_with_fin = pos.merge(
    fin,
    left_on=["stock_code", "t_minus_2_year"],
    right_on=["stock_code", "year"],
    how="left"
)

matched = pos_with_fin["total_assets"].notna().sum()
unmatched = len(pos) - matched

print(f"正样本总数:       {len(pos)} 家")
print(f"成功匹配财务数据: {matched} 家")
print(f"未匹配（数据缺失）: {unmatched} 家")

if unmatched > 0:
    miss = pos_with_fin[pos_with_fin["total_assets"].isna()][["stock_code", "stock_name", "t_minus_2_year"]]
    print(f"\n未匹配的公司：")
    print(miss.to_string(index=False))

正样本总数:       138 家
成功匹配财务数据: 138 家
未匹配（数据缺失）: 0 家


In [12]:
# 只保留有财务数据的正样本，丢掉财务缺失的
pos_valid = pos_with_fin[pos_with_fin["total_assets"].notna()].copy().reset_index(drop=True)
pos_valid["label"] = 1   # 正样本标签 = 1

print(f"有效正样本: {len(pos_valid)} 家（丢弃 {len(pos) - len(pos_valid)} 家财务缺失的）")

有效正样本: 138 家（丢弃 0 家财务缺失的）


## Step 2 — 构建 1:1 配对函数

配对规则（按优先级）：
1. **同申万行业**（`sw_industry` 相同）
2. **同年份**（`year` 相同，即用同一年的财务数据比较）
3. **总资产规模 ±30%**（`total_assets` 在正样本的 0.7x ~ 1.3x 之间）
4. **不在正样本名单里**（排除 138 家 ST 公司）
5. **随机抽 1 家**（固定随机种子，保证结果可复现）

如果某个行业候选不足，放宽到 ±50%；再不够，记录下来人工处理。

In [13]:
# 正样本的股票代码集合，用于排除
st_codes = set(pos["stock_code"].tolist())

def find_match(row: pd.Series, fin_df: pd.DataFrame, rng: np.random.Generator) -> pd.Series | None:
    """
    为单个正样本找一个配对的负样本。
    row:    正样本的一行数据
    fin_df: 全 A 股财务数据
    rng:    随机数生成器（固定种子，保证可复现）
    返回: 一行负样本数据，或 None（找不到时）
    """
    target_assets  = row["total_assets"]
    target_year    = row["year"]         # T-2 年
    target_industry = row["sw_industry"]

    # 候选池：同年份 + 同行业 + 不是正样本
    pool = fin_df[
        (fin_df["year"] == target_year) &
        (fin_df["sw_industry"] == target_industry) &
        (~fin_df["stock_code"].isin(st_codes))
    ].copy()

    # 先尝试 ±30% 的规模范围
    for scale in [0.30, 0.50, 1.00]:   # 逐步放宽，最后 1.00 = 不限规模
        size_pool = pool[
            (pool["total_assets"] >= target_assets * (1 - scale)) &
            (pool["total_assets"] <= target_assets * (1 + scale))
        ]
        if len(size_pool) > 0:
            return size_pool.sample(1, random_state=None).iloc[0]

    return None   # 实在找不到

print("配对函数已定义，准备执行配对...")

配对函数已定义，准备执行配对...


## Step 3 — 执行 1:1 配对

In [14]:
np.random.seed(42)   # 固定随机种子，保证每次跑结果一样
rng = np.random.default_rng(42)

neg_rows   = []   # 成功配对的负样本
failed_pos = []   # 配对失败的正样本（记录下来）

for _, row in pos_valid.iterrows():
    match = find_match(row, fin, rng)
    if match is not None:
        neg_rows.append(match)
    else:
        failed_pos.append(row["stock_code"])

print(f"配对成功: {len(neg_rows)} 家负样本")
print(f"配对失败: {len(failed_pos)} 家正样本（{failed_pos}）")

# 把负样本整理成 DataFrame
neg = pd.DataFrame(neg_rows).reset_index(drop=True)
neg["label"] = 0   # 负样本标签 = 0

配对成功: 138 家负样本
配对失败: 0 家正样本（[]）


## Step 4 — 合并正负样本，保存建模数据集

只保留财务特征列 + `stock_code` + `year` + `label`，去掉不用于建模的元数据列。

In [15]:
# 财务特征列（从 02_financials_long 里来的所有数值列）
feature_cols = [
    "ar_turnover", "asset_turnover", "cash_interest_coverage",
    "current_asset_turnover", "current_ratio", "debt_to_asset",
    "debt_to_equity", "ebit", "eps", "fixed_asset_turnover",
    "gross_margin", "inventory_turnover", "net_assets_growth",
    "net_margin", "net_profit_growth", "ocf_per_share",
    "operating_cash_flow", "operating_income_ratio", "operating_margin",
    "quick_ratio", "retained_earnings", "revenue_growth", "roa", "roe",
    "top10_holders_pct", "total_assets", "total_assets_growth",
    "total_revenue", "working_capital", "audit_opinion_code",
    "is_non_standard_audit",
]

# merge 后 stock_name 可能变成 stock_name_x（来自 pos）和 stock_name_y（来自 fin）
# 统一处理：优先用 stock_name，没有就用 stock_name_x
if "stock_name" not in pos_valid.columns and "stock_name_x" in pos_valid.columns:
    pos_valid = pos_valid.rename(columns={"stock_name_x": "stock_name"})
if "stock_name_y" in pos_valid.columns:
    pos_valid = pos_valid.drop(columns=["stock_name_y"])

keep_cols = ["stock_code", "stock_name", "sw_industry", "year", "label"] + feature_cols

# 正样本：取相应的列
pos_final = pos_valid[keep_cols].copy()

# 负样本：只保留财务数据里有的列（neg 来自 fin，不含 pos 的专属列）
neg_cols_available = [c for c in keep_cols if c in neg.columns]
neg_final = neg[neg_cols_available].copy()

# 合并
dataset = pd.concat([pos_final, neg_final], ignore_index=True)
dataset = dataset.sample(frac=1, random_state=42).reset_index(drop=True)

# 保存
output_path = PROCESSED_DIR / "03_modeling_dataset.csv"
dataset.to_csv(output_path, index=False, encoding="utf-8-sig")

print(f"✓ 已保存: {output_path}")
print(f"  总行数: {len(dataset)}（正样本 {(dataset['label']==1).sum()} + 负样本 {(dataset['label']==0).sum()}）")
print(f"  总列数: {len(dataset.columns)}")
dataset.head(3)

✓ 已保存: /Users/aa00551/Desktop/Projects/A-Share-ST-Risk-Predictor/data/processed/03_modeling_dataset.csv
  总行数: 276（正样本 138 + 负样本 138）
  总列数: 36


,stock_code,stock_name,sw_industry,year,label,ar_turnover,asset_turnover,cash_interest_coverage,current_asset_turnover,current_ratio,...,revenue_growth,roa,roe,top10_holders_pct,total_assets,total_assets_growth,total_revenue,working_capital,audit_opinion_code,is_non_standard_audit
0,600107.SH,*ST尔雅,服装家纺,2023,1,8.0119,0.3725,-351.273967,0.5754,1.9968,...,5.4988,-0.8232,7.2134,31.46,1.050793e+09,-24.2453,4.539974e+08,4.377746e+08,2.0,1
1,002157.SZ,正邦科技,养殖业,2021,1,147.0259,0.9009,-175.094691,2.0411,0.4784,...,-3.0429,-33.8578,-184.0133,59.07,4.656700e+10,-21.4186,4.767022e+10,-1.646885e+10,0.0,0
2,300621.SZ,维业股份,装修装饰Ⅱ,2023,0,6.9206,1.2281,-79.359412,1.2836,1.0321,...,4.9667,2.0393,0.1820,56.28,1.312649e+10,7.9253,1.552896e+10,3.919591e+08,0.0,0


## Step 5 — 质量检查报告

In [16]:
missing_pct = dataset[feature_cols].isna().mean() * 100

print("=" * 56)
print("           建模数据集质量报告")
print("=" * 56)
print(f"总样本数:   {len(dataset)} 行")
print(f"正样本:     {(dataset['label']==1).sum()} 家  负样本: {(dataset['label']==0).sum()} 家")
print(f"特征列数:   {len(feature_cols)} 个")

print("\n按年份分布:")
print(dataset.groupby(["year","label"]).size().unstack(fill_value=0).to_string())

print("\n缺失率 >20% 的特征（需要在建模前处理）:")
high_missing = missing_pct[missing_pct > 20].sort_values(ascending=False)
if len(high_missing) > 0:
    for col, pct in high_missing.items():
        print(f"  {col}: {pct:.1f}%")
else:
    print("  无（所有特征缺失率均 ≤20%）")

print("\n是否有重复的负样本（同一家公司被配对两次）:")
dup_neg = dataset[dataset["label"]==0]["stock_code"].duplicated().sum()
print(f"  重复负样本数: {dup_neg}")

print("=" * 56)
print("✓ 建模数据集准备完毕，下一步: 04_modeling.ipynb")

           建模数据集质量报告
总样本数:   276 行
正样本:     138 家  负样本: 138 家
特征列数:   31 个

按年份分布:
label   0   1
year         
2021   24  24
2022   43  43
2023   71  71

缺失率 >20% 的特征（需要在建模前处理）:
  operating_income_ratio: 46.7%

是否有重复的负样本（同一家公司被配对两次）:
  重复负样本数: 7
✓ 建模数据集准备完毕，下一步: 04_modeling.ipynb
